In [ ]:
from coxmodel import CoxPH
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold, GridSearchCV
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wandb
from scipy.stats import randint, uniform

# Read datasets

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive # type: ignore
    drive.mount('/content/drive')
    train_df_csv = "/content/drive/MyDrive/bachelor/nacc_data_2025.csv"
else:
    train_df_csv = "./train_preprocessed.csv"

In [ ]:
train_df = pd.read_csv(train_df_csv, delimiter=',')

In [ ]:
train_df.shape

## Compute X and y and calculate weights

In [ ]:
train_X, train_y = train_df.drop(columns=[SURVIVAL_TIME_COL, SURVIVAL_EVENT_COL])
y = build_survival_target(df)
case_weights = compute_sample_weight(
  class_weight='balanced',
  y=train_y["EVENT_MCI"]
)

# Initialize wandb

In [ ]:
features_num = train_df.shape[1]
n_samples = train_df.shape[0]

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="semariik",
    # Set the wandb project where this run will be logged.
    project="survival-analysis-mci",
    # Track hyperparameters and run metadata.
    config={
        "model":            "RandomSurvivalForest",
        "tuning_strategy":  "RandomizedSearchCV",
        "n_iter":           10,
        "cv_folds":         5,
        "n_samples":        n_samples,
        "n_features":       features_num,
        "scoring":          "concordance_index",
        "dataset":          "MCI",
        "param_space": {
            "num_trees":       "randint(1, 2001)",
            "mtry":            "['sqrt', 'log2']",
            "min_node_size":   "n^x transformation",
            "replace":         "[True, False]",
            "sample_fraction": "uniform(0.1, 0.9)",
        }
    },
)

# Parameters and constents definition

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
duration_col = "TIME"
event_col    = "EVENT_MCI"
lambdas      = 2 ** np.linspace(-10, 10, 50)

In [ ]:
param_grid = {
    'penalizer': lambdas,  # lambda
    'l1_ratio':  [0.0, 0.25, 0.5, 0.75, 1.0],    # alpha
}

# Cox model tunning

## Tune parameters with GridSearchCV

In [ ]:
grid_search = GridSearchCV(
    estimator=CoxPH(compute_weights=True),
    param_grid=param_grid,
    cv=cv.split(train_X, train_y[event_col].astype(int)),
    scoring=None,   # uses model's score() method
    refit=False,    # refit manually after finding optimal lambda
)


In [ ]:
grid_search.fit(train_X, train_y, sample_weight=case_weights)

## Asses overfitting and underfitting and select optimal lambda

Create error curves for cross validation and train. Check that optimally selected parameters are the optimal point,w here no overfit no undefit. Reselect better lambda, in case of need by graph

In [ ]:
train_errors = []
val_errors   = []
val_stds     = []

for penalizer in lambdas:
    fold_train = []
    fold_val   = []

    for train_idx, val_idx in cv.split(train_X, train_y[event_col].astype(int)):
        df_train = train_X.iloc[train_idx]
        df_val   = train_X.iloc[val_idx]

        model = CoxPH(penalizer=penalizer, l1_ratio=0.0, compute_weights=True)
        model.fit(df_train, duration_col=duration_col, event_col=event_col)

        c_index_train = model.score(df_train, duration_col=duration_col, event_col=event_col)
        c_index_val   = model.score(df_val, duration_col=duration_col, event_col=event_col)

        fold_train.append(1 - c_index_train)
        fold_val.append(1 - c_index_val)

    train_errors.append(np.mean(fold_train))
    val_errors.append(np.mean(fold_val))
    val_stds.append(np.std(fold_val) / np.sqrt(5))

    wandb.log({
        "cox/lambda":      penalizer,
        "cox/log_lambda":  np.log(penalizer),
        "cox/train_error": train_errors[-1],
        "cox/val_error":   val_errors[-1],
        "cox/gap":         val_errors[-1] - train_errors[-1],
    })